In [8]:
import sys
import polars as pl

from Python import statcast

In [ ]:
sys.path.insert(0, "/Users/cbkaplinger/Desktop/Desktop - Cameron MacBook Pro/MLB-Props/src/Python")
years = [2025]

In [3]:
print(statcast.STRIKEOUT_EVENTS)
print(statcast.NON_PA_EVENTS)
print(statcast.WHIFF_DESCRIPTIONS)
print(statcast.SWING_DESCRIPTIONS, statcast.CONTACT_DESCRIPTIONS)

frozenset({'strikeout', 'strikeout_double_play'})
frozenset({'batter_interference', 'pickoff_caught_stealing_2b', 'pickoff_1b', 'caught_stealing_2b', 'runner_double_play', 'passed_ball', 'pickoff_caught_stealing_3b', 'other_out', 'caught_stealing_home', 'stolen_base_2b', 'pickoff_3b', 'caught_stealing_3b', 'stolen_base_home', 'stolen_base_3b', 'pickoff_caught_stealing_home', 'wild_pitch', 'pickoff_2b'})
frozenset({'swinging_strike', 'foul_tip', 'swinging_strike_blocked'})
frozenset({'swinging_strike', 'hit_into_play', 'foul', 'foul_tip', 'swinging_strike_blocked'}) frozenset({'hit_into_play', 'foul'})


In [4]:
raw = statcast.load_statcast_years(years, columns=None)
raw.shape, raw.columns

((706571, 119),
 ['pitch_type',
  'game_date',
  'release_speed',
  'release_pos_x',
  'release_pos_z',
  'player_name',
  'batter',
  'pitcher',
  'events',
  'description',
  'spin_dir',
  'spin_rate_deprecated',
  'break_angle_deprecated',
  'break_length_deprecated',
  'zone',
  'des',
  'game_type',
  'stand',
  'p_throws',
  'home_team',
  'away_team',
  'type',
  'hit_location',
  'bb_type',
  'balls',
  'strikes',
  'game_year',
  'pfx_x',
  'pfx_z',
  'plate_x',
  'plate_z',
  'on_3b',
  'on_2b',
  'on_1b',
  'outs_when_up',
  'inning',
  'inning_topbot',
  'hc_x',
  'hc_y',
  'tfs_deprecated',
  'tfs_zulu_deprecated',
  'umpire',
  'sv_id',
  'vx0',
  'vy0',
  'vz0',
  'ax',
  'ay',
  'az',
  'sz_top',
  'sz_bot',
  'hit_distance_sc',
  'launch_speed',
  'launch_angle',
  'effective_speed',
  'release_spin_rate',
  'release_extension',
  'game_pk',
  'fielder_2',
  'fielder_3',
  'fielder_4',
  'fielder_5',
  'fielder_6',
  'fielder_7',
  'fielder_8',
  'fielder_9',
  'releas

In [5]:
pa = statcast.plate_appearances(raw)
pa.shape
pa.select(["game_pk","at_bat_number"]).is_duplicated().sum()  # should be 0

0

In [6]:
flagged = statcast.add_event_flags(raw)
flagged.select(["is_pa","is_k","is_bb","is_hbp","is_hr","is_hit","is_whiff","is_called_strike"]).sum()

is_pa,is_k,is_bb,is_hbp,is_hr,is_hit,is_whiff,is_called_strike
u32,u32,u32,u32,u32,u32,u32,u32
181396,40894,14301,1998,5426,39539,85393,115390


In [10]:
counts = disc.group_by("pitcher").agg(
    pl.len().alias("Pitches"),
    pl.col("is_whiff").sum().alias("Whiffs"),
    *statcast.discipline_count_exprs(),
)
rates = statcast.add_plate_discipline_rates(counts)

In [11]:
woba_check = pa.group_by("batter").agg(
    pl.len().alias("PA"),
    statcast.woba_agg(),
    statcast.xwoba_agg(),
)
woba_check.sort("PA", descending=True).head(10)

batter,PA,wOBA,xwOBA
i64,u32,f64,f64
680776,733,0.375,0.340699
683002,716,0.390126,0.374387
660271,716,0.4483,0.446886
665742,714,0.431056,0.462682
543760,713,0.318856,0.314567
677951,706,0.425072,0.414388
592450,704,0.496481,0.49069
678662,693,0.335944,0.294837
665489,693,0.412077,0.415388


In [12]:
k_rates = statcast.batter_k_rate(pa, min_pa=200)
k_rates.head(10)

batter,PA,K,k_rate
i64,u32,u32,f64
666181,388,154,0.396907
608336,258,101,0.391473
642350,448,170,0.379464
669357,404,151,0.373762
669234,230,85,0.369565
670242,261,95,0.363985
694388,258,93,0.360465
669065,210,74,0.352381
572191,302,105,0.347682


In [13]:
identity_check = disc.select(
    (pl.col("is_swing").sum() == (pl.col("is_whiff").sum() + pl.col("is_contact").sum())).alias("swings_eq_whiffs_plus_contacts")
)
identity_check

swings_eq_whiffs_plus_contacts
bool
true
